# Day 8 · Exercise 3: Structured Classification Result

**What you'll build:** `classify_with_confidence(text: str, labels: list[str], model: str) -> dict` — a classifier that asks the model to return a JSON object containing a label, a short reasoning string, and a numeric confidence score, then parses and validates it with the Day-4 Pydantic pattern.

**Why it matters:** A bare label tells you *what* the model decided; structured output tells you *why* and *how sure* — the two extra fields that unlock auditing, confidence-based routing, and meaningful debugging.

## Your Implementation

In [ ]:
import json
import logging
import ollama
from pydantic import BaseModel, Field, ValidationError

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(name)s | %(message)s",
)
logger = logging.getLogger(__name__)


class ClassificationResult(BaseModel):
    """Structured output from the classifier.

    Attributes:
        label:      The chosen category, exactly as it appears in the label list.
        reasoning:  One-to-two sentence explanation of the choice.
        confidence: Self-reported confidence from 0.0 (uncertain) to 1.0 (certain).
    """

    label: str = Field(description="Exactly one label from the provided list")
    reasoning: str = Field(description="Brief explanation of why this label was chosen")
    confidence: float = Field(
        description="Confidence score between 0.0 (uncertain) and 1.0 (certain)"
    )


def classify_with_confidence(text: str, labels: list[str], model: str) -> dict:
    """Classify text and return a structured result with label, reasoning, and confidence.

    Builds a schema-guided system prompt that embeds the ClassificationResult
    JSON schema and a concrete example output, calls ollama.chat with
    format='json' to enable grammar-constrained sampling, then parses the
    response with ClassificationResult.model_validate_json().  If parsing
    fails a graceful fallback dict is returned so the caller never receives
    an unexpected exception.

    Args:
        text:   The text to classify.
        labels: List of valid output labels (e.g. ["positive", "negative"]).
        model:  Ollama model name (e.g. "llama3.2").

    Returns:
        A dict with keys ``label`` (str), ``reasoning`` (str), and
        ``confidence`` (float).  On parse failure returns
        ``{"label": "unknown", "reasoning": "parse error", "confidence": 0.0}``.

    Example:
        result = classify_with_confidence(
            "I love this product!",
            labels=["positive", "negative", "neutral"],
            model="llama3.2",
        )
        # result == {"label": "positive", "reasoning": "...", "confidence": 0.95}
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(classify_with_confidence), 'classify_with_confidence is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # later checks would NameError — stop here

    # Check 2: returns a dict with the three required keys
    try:
        result = classify_with_confidence(
            "I absolutely loved the movie — incredible storytelling!",
            labels=["positive", "negative", "neutral"],
            model="llama3.2",
        )
        assert isinstance(result, dict), f'expected dict, got {type(result).__name__}'
        for key in ("label", "reasoning", "confidence"):
            assert key in result, f'missing key: {key!r}'
        print(f'{_PASS} Check 2/{total}: returns a dict with label, reasoning, confidence')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return  # confidence check below depends on a valid result

    # Check 3: label is one of the supplied labels
    try:
        valid_labels = ["positive", "negative", "neutral"]
        result2 = classify_with_confidence(
            "The service was slow and the staff were rude.",
            labels=valid_labels,
            model="llama3.2",
        )
        assert result2["label"] in valid_labels, \
            f'label {result2["label"]!r} not in {valid_labels}'
        print(f'{_PASS} Check 3/{total}: label is one of the supplied labels')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: confidence is a float between 0.0 and 1.0
    try:
        conf = result["confidence"]
        assert isinstance(conf, float), f'confidence must be float, got {type(conf).__name__}'
        assert 0.0 <= conf <= 1.0, f'confidence {conf} is outside [0.0, 1.0]'
        print(f'{_PASS} Check 4/{total}: confidence is a float in [0.0, 1.0] (got {conf:.2f})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

Right now `classify_with_confidence` returns a plain `dict`. On Day 9 you will work with typed objects throughout, so here is a preview: change the return type to return a `ClassificationResult` object directly instead of calling `.model_dump()`. Then update the function signature to `-> ClassificationResult` and update the fallback to return `ClassificationResult(label="unknown", reasoning="parse error", confidence=0.0)`. Notice how the caller now accesses fields as attributes (`result.confidence`) rather than dict keys (`result["confidence"]`). Which style do you prefer, and why?

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import json
import logging
import ollama
from pydantic import BaseModel, Field, ValidationError

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(name)s | %(message)s",
)
logger = logging.getLogger(__name__)


class ClassificationResult(BaseModel):
    label: str = Field(description="Exactly one label from the provided list")
    reasoning: str = Field(description="Brief explanation of why this label was chosen")
    confidence: float = Field(
        description="Confidence score between 0.0 (uncertain) and 1.0 (certain)"
    )


_FALLBACK = {"label": "unknown", "reasoning": "parse error", "confidence": 0.0}

_SYSTEM_PROMPT = """You are a text classifier.

Classify the text into exactly one of these labels: {labels}

Reply with a single JSON object that matches this schema exactly.
No prose, no markdown, no explanation outside the JSON.

Schema:
{schema}

Example output:
{{"label": "negative", "reasoning": "The review expresses clear dissatisfaction.", "confidence": 0.92}}"""


def classify_with_confidence(text: str, labels: list[str], model: str) -> dict:
    """Classify text and return a structured result with label, reasoning, and confidence."""
    schema_str = json.dumps(ClassificationResult.model_json_schema(), indent=2)
    label_str = ", ".join(labels)
    system_msg = _SYSTEM_PROMPT.format(labels=label_str, schema=schema_str)

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Text to classify:\n{text}"},
    ]

    logger.debug("classify_with_confidence() | text=%r", text[:60])
    response = ollama.chat(model=model, messages=messages, format="json")
    raw = response["message"]["content"]

    try:
        result = ClassificationResult.model_validate_json(raw)
        if result.label.lower() not in [lb.lower() for lb in labels]:
            logger.warning("Label %r not in allowed set %s", result.label, labels)
            return _FALLBACK
        logger.info(
            "Classified as %r (confidence=%.2f)", result.label, result.confidence
        )
        return result.model_dump()
    except ValidationError as e:
        logger.error("Pydantic validation failed: %s | raw=%r", e, raw[:200])
        return _FALLBACK
```

**Why this works:** Embedding `ClassificationResult.model_json_schema()` in the system prompt tells the model the exact JSON shape to produce, and passing `format='json'` to `ollama.chat` applies grammar-constrained sampling at the token level — together they make malformed JSON rare rather than common. After `model_validate_json` succeeds, the label-membership check guards against the model returning a syntactically valid JSON label that is not actually in the allowed set — if it happens, a warning is logged and the fallback is returned, making Check 3 robust to model drift. Wrapping `model_validate_json` in a `try/except ValidationError` block means the rare parse failures are logged and handled gracefully rather than crashing the caller. Calling `.model_dump()` on the validated `ClassificationResult` converts it back to a plain `dict`, which is the return type the function signature advertises.
</details>